### Multiclass image classification

> *We've just been through a bunch of the following steps with a binary classification problem (pizza vs steak), now we're going to step things up a notch with 10 classes of food (multiclass classification)*

1. Become one with the data
2. Preprocess the data (get it ready for a model)
3. Create a model (start with a baseline)
4. Fit the model (overfit it to make sure it works)
5. Evaluate the model
6. Adjust different hyperparameters and improve the model (try to beat the baseline/reduce overfitting)
7. Repeat until satisfied

In [ ]:
import tensorflow as tf

### 1. Importing and becoming one with the data

In [ ]:
import zipfile

!curl -o ../../data/10_food_classes_all_data.zip https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_all_data.zip


In [ ]:
# unzip our data
zip_ref = zipfile.ZipFile('../../data/10_food_classes_all_data.zip', 'r')
zip_ref.extractall()
zip_ref.close()

In [ ]:
# walk through 10 classes of food images data
import os

for dirpath, dirname, filename in os.walk('10_food_classes_all_data'):
    print(f'There are {len(dirname)} directories and {len(filename)} images in {dirpath}')

In [ ]:
# setup train and test directories
train_dir = '10_food_classes_all_data/train/'
test_dir = '10_food_classes_all_data/test/'

In [ ]:
# let's get the class names
import pathlib
import numpy as np

data_dir = pathlib.Path(train_dir)
class_names = np.array(sorted([item.name for item in data_dir.glob('*')]))

print(class_names)

In [ ]:
# visualize visulaize visualize
# Lets visualize our images
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random

def view_random_image(target_dir, target_class):
    '''Setup the target directories (we'll view the images from here)'''
    target_folder = target_dir+target_class
    
    # get a random image path
    random_image = random.sample(os.listdir(target_folder), 1)
    print(random_image)
    
    # read in the image and plot it 
    img = mpimg.imread(target_folder + "/" + random_image[0])
    plt.imshow(img)
    plt.title(target_class)
    plt.axis("off")
    
    print(f"Image shape: {img.shape}") # shwo the shape of the image
    
    return img

In [ ]:
# view a random image from the training dataset
img = view_random_image(target_dir=train_dir,
                        target_class=random.choice(class_names))

### 2. Preprocess the data

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# rescale 
train_datagen = ImageDataGenerator(rescale=1/255.)
test_datagen = ImageDataGenerator(rescale=1/255.)

# load data in from directories and turn it into batches
train_data = train_datagen.flow_from_directory(train_dir,
                                               target_size=(224, 224),
                                               batch_size=32,
                                               class_mode='categorical')

test_data = test_datagen.flow_from_directory(test_dir,
                                               target_size=(224, 224),
                                               batch_size=32,
                                               class_mode='categorical')

### 3. Create the model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense, Activation

# create our model
model = Sequential([
    Conv2D(10, 3, input_shape=(224, 224, 3)),
    Activation(activation='relu'),
    Conv2D(10, 3, activation='relu'),
    MaxPool2D(),
    Conv2D(10, 3, activation='relu'),
    Conv2D(10, 3, activation='relu'),
    MaxPool2D(),
    Flatten(),
    Dense(10, activation='softmax')    
])

# compile the model
model.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(),
              metrics=['accuracy'])

In [ ]:
# fit the model
history = model.fit(train_data,
                    epochs=5,
                    steps_per_epoch=len(train_data),
                    validation_data=test_data,
                    validation_steps=len(test_data))

### 4. Evaluate the model

In [ ]:
model.evaluate(test_data)

In [ ]:
# check out the model loss curves
# plot validation and training curves seperately
def plot_loss_curves(history):
    ''' 
    Returns seperate loss curves for traning and validation metrics.
    '''
    loss = history.history['loss']
    val_loss = history.history["val_loss"]
    
    accuracy = history.history['accuracy']
    val_accuracy = history.history["val_accuracy"]
    
    epochs = range(len(history.history["loss"])) # how many epochs did we run for
    
    # plot loss
    plt.plot(epochs, loss, label="training loss")
    plt.plot(epochs, val_loss, label="val_loss")
    plt.xlabel("epochs")
    plt.title("loss")
    plt.legend()
    
    # plot accuracy
    plt.figure()
    plt.plot(epochs, accuracy, label="training accuracy")
    plt.plot(epochs, val_accuracy, label="val_accuracy")
    plt.xlabel("epochs")
    plt.title("accuracy")
    plt.legend()

In [ ]:
plot_loss_curves(history)

What do these loss curves tell us?

Well... it seems our model is **overfitiing** the training set quiet badly... in other words its getting great results on trining data but fails to generalize well to unseen data and performs poorly on test dataset.

### 5. Adjust the model Hyperparameter (to beat the baseline/reduce overfitting)

Due to its performance on the training data, it's clear our model is learning something...

However, it's not generalizing well to unseen data (overfitting)

Let's try and fix overfitting by...

* **Get more data** - having more data gives a model more opportunity to learn diverse patterns...
* **Simplify the model** - if our current model is overfitting the data, it means it may be too complicated model, one way to simplify a model is to: reduce # of layers or reduce # hidden units in layers.
* **Use data augmentation** - data augmentation manipulates the training data in such a way to ad moree diversity to it (without altering the original data).
* **Use transfer learning** - transfer learning leverages the patterns another model has learned on similar data to your own and aloows you to use those patterns on your own dataset.

In [ ]:
model.summary()

In [ ]:
# simplify the model first
# let's try to remove 2 convolutional layers
model_1 = Sequential([
    Conv2D(10, 3, input_shape=(224, 224, 3)),
    Activation(activation='relu'),
    MaxPool2D(),
    Conv2D(10, 3, activation='relu'),
    MaxPool2D(),
    Flatten(),
    Dense(10, activation='softmax')
])

# compile the model
model_1.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(),
              metrics=['accuracy'])

In [ ]:
model_1.summary()

In [ ]:
# fit the model
history_1 = model_1.fit(train_data,
                    epochs=5,
                    steps_per_epoch=len(train_data),
                    validation_data=test_data,
                    validation_steps=len(test_data))

In [ ]:
# check out the loss curves
plot_loss_curves(history_1)

Looks like our "Simplify the model" experiment didin't work... the accuracy went down and overfittin g continue.

How about try data augmentation

### Trying to reduce overfitting with data augmentation

Lets's try and improve our model's result by using augmented training data...

Ideally we want to:
* Reduce overfitting (get the training and validation loss curves closer)
* Improve validation accuracy

In [ ]:
# create an augmented data generator instance
train_datagen_augmented = ImageDataGenerator(rescale=1/255.,
                                          rotation_range=0.2,
                                          width_shift_range=0.2,
                                          height_shift_range=0.2,
                                          zoom_range=0.2,
                                          horizontal_flip=True)

train_data_augmented = train_datagen_augmented.flow_from_directory(train_dir,
                                                                   target_size=(224,224),
                                                                   batch_size=32,
                                                                   class_mode='categorical')
                                                                   

In [ ]:
# lets create another model but this time fit on augmented data
model_2 = tf.keras.models.clone_model(model)

# compile the cloned model (using the same setup as previous models)
model_2.compile(loss='categorical_crossentropy',
                optimizer=tf.keras.optimizers.Adam(),
                metrics=['accuracy'])

In [ ]:
model_2.summary()

In [ ]:
# fit the model
history_2 = model_2.fit(train_data_augmented,
                    epochs=5,
                    steps_per_epoch=len(train_data_augmented),
                    validation_data=test_data,
                    validation_steps=len(test_data))

In [ ]:
model_2.evaluate(test_data)

In [ ]:
# plot the loss curves
plot_loss_curves(history_2)

### Repeat until satisfied

We could keep going here... continuously tring to bring our loss curves closer together and trying to improve the validation/test accuracy.

How?

By running lots of experiments, namely:
- restructuring our model's architecture (increasing layers/hidden units)
- adjust the learning rates
- try different methods of data augmentation (adjust the hyperparameters in our ImageDataGenerator instances)
- training for longer (e.g for 10 epochs instead of 5)
- try **transfer learning**

### Making prediction with our trained model

Let's use our trained model to make some predictions on our own custom Images!

In [ ]:
class_names

In [ ]:
# download some custom images
!curl -L https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/images/03-sushi.jpeg -o 03-sushi.jpeg

In [ ]:
# amke a predict
# create a function to import an image and resize it to be able to be use with our model
def load_and_prep_image(filename, img_shape=224):
    ''' 
    Reads the image from filename, turns it into a tensor and reshape it to 
    (img_shape, img_shape, color_channel).
    '''
    img = tf.io.read_file(filename)
    
    # decode the read file into tensor
    img = tf.image.decode_image(img)
    
    # resize the image
    img = tf.image.resize(img, size=[img_shape, img_shape])
    
    # rescale the image (get all values between 0 & 1)
    img = img/255.
    
    return img

# function to predict and plot
def pred_and_plot(model, filename, class_names=class_names):
    ''' 
    Imports and image located at filename, makes a prediction with model and plots the image with the
    predicted class as the title.
    '''
    # Import the target image abd prprocess it
    img = load_and_prep_image(filename)
    
    # make a prediction
    pred = model.predict(tf.expand_dims(img, axis=0))
    
    # Add logic for multiclass
    if len(pred[0]) > 1:
        pred_class = class_names[tf.argmax(pred[0])]
    else:
        pred_class = class_names[int(tf.round(pred[0]))]

    
    # plot the image and predicted class
    plt.imshow(img)
    plt.title(f"Prediction: {pred_class}")
    plt.axis(False);

In [ ]:
# making some prediction
pred_and_plot(model=model_2,
              filename='03-pizza-dad.jpeg',
              class_names=class_names)

In [ ]:
pred_and_plot(model=model_2,
              filename='03-sushi.jpeg',
              class_names=class_names)

### Savingand loading our model

In [ ]:
# save a model
model_2.save('../../assets/saved_trained_model_2.keras')

In [ ]:
# load the trained model and evaluate
loaded_model_2 = tf.keras.models.load_model('../../assets/saved_trained_model_2.keras')
loaded_model_2.evaluate(test_data)

In [ ]:
# compare tge loaded model with our original model
model_2.evaluate(test_data)

## Transfer Learning